# Lab 7: Measuring Earthquake Slip with Optical Pixel Tracking

> **Colab note:** This notebook is designed to run on **Google Colab**. [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amtseismo/EPS166/blob/main/notebooks/04_optical_pixel_tracking.ipynb)

## Introduction

On July 4 and 5, 2019, a M6.4 foreshock and M7.1 mainshock ruptured the surface of the Mojave Desert near Ridgecrest, California. The NW-trending right-lateral ruptures produced meters of surface displacement across more than 50 km — one of the largest surface ruptures in California in decades.

In this lab you will compare Sentinel-2 satellite images acquired before and after the earthquakes and use **sub-pixel cross-correlation** (pixel tracking) to measure the surface displacement field. You will extract a fault-perpendicular displacement profile, estimate the total fault slip, and compare your result to field and GNSS observations.

Along the way you will see directly why optical pixel tracking measures what InSAR cannot: the fault-parallel horizontal motion that is nearly invisible in satellite radar line-of-sight.

**Data:**  
Pre-event Sentinel-2 Band 8 (NIR, 10 m): acquired 2019-06-24  
Post-event Sentinel-2 Band 8 (NIR, 10 m): acquired 2019-07-14  
Course copies hosted on the EPS 166 GitHub repository.

## Learning objectives

By the end, you will be able to:

- load and visualize pre- and post-event optical satellite imagery
- explain why Band 8 (NIR) is preferred for pixel tracking over desert terrain
- apply chip-based sub-pixel cross-correlation to estimate surface displacement
- extract and interpret a fault-perpendicular displacement profile
- estimate total co-seismic slip from the profile step
- explain why InSAR is insensitive to this signal given the fault geometry

## Notebook Outline
- [Part I: Setup and data loading](#part-i-setup-and-data-loading)
- [Part II: Visualize the pre- and post-event images](#part-ii-visualize-the-pre--and-post-event-images)
- [Part III: Coregistration check](#part-iii-coregistration-check)
- [Part IV: Sub-pixel cross-correlation](#part-iv-sub-pixel-cross-correlation)
- [Part V: Displacement profile across the fault](#part-v-displacement-profile-across-the-fault)
- [Part VI: Estimating total slip](#part-vi-estimating-total-slip)
- [Part VII: Why InSAR misses this signal](#part-vii-why-insar-misses-this-signal)
- [Part VIII: Extension — 2D displacement map](#part-viii-extension--2d-displacement-map)
- [Synthesis](#synthesis)
- [Summary](#summary)


## Part I: Setup and data loading

Install `rasterio` for reading GeoTIFFs, then import the packages used throughout the lab.


In [ ]:
%pip install -q rasterio

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import rasterio
from rasterio.plot import show
from skimage.registration import phase_cross_correlation
from scipy.ndimage import uniform_filter
import urllib.request
import os

plt.rcParams.update({
    "figure.dpi": 120,
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.grid": True,
    "grid.alpha": 0.25,
})

### Download the imagery

Both scenes are pre-clipped Sentinel-2 Band 8 (NIR, 10 m resolution) GeoTIFFs covering a ~50 × 50 km window centered on the Ridgecrest rupture zone. They are hosted on the course GitHub repository so no API credentials are needed.


In [ ]:
BASE_URL = (
    "https://github.com/amtseismo/EPS166/"
    "raw/main/datasets/"
)

files = {
    "pre":  "ridgecrest_pre_B08.tif",
    "post": "ridgecrest_post_B08.tif",
}

for key, fname in files.items():
    if not os.path.exists(fname):
        print(f"Downloading {fname} ...")
        urllib.request.urlretrieve(BASE_URL + fname, fname)
    else:
        print(f"{fname} already present.")

print("\nDone.")

### Load the images

Read each GeoTIFF into a NumPy array and extract the geotransform so we can plot in real-world coordinates (UTM meters).


In [ ]:
def load_band(path):
    """
    Load a single-band GeoTIFF.

    Parameters
    ----------
    path : str
        File path to the GeoTIFF.

    Returns
    -------
    arr : numpy.ndarray  (rows, cols), float32
        Image data, nodata pixels set to NaN.
    transform : affine.Affine
        Affine geotransform (pixel → map coordinates).
    crs : rasterio.crs.CRS
        Coordinate reference system.
    extent : list
        [left, right, bottom, top] in map units for imshow.
    """
    with rasterio.open(path) as src:
        arr = src.read(1).astype(np.float32)
        nodata = src.nodata
        transform = src.transform
        crs = src.crs
        left, bottom, right, top = src.bounds
    if nodata is not None:
        arr[arr == nodata] = np.nan
    extent = [left, right, bottom, top]
    return arr, transform, crs, extent


pre,  tf_pre,  crs, extent = load_band(files["pre"])
post, tf_post, _,   _      = load_band(files["post"])

pixel_size_m = abs(tf_pre.a)   # pixel size in metres (should be 10 m)

print(f"Image size  : {pre.shape[0]} rows × {pre.shape[1]} cols")
print(f"Pixel size  : {pixel_size_m:.0f} m")
print(f"CRS         : {crs.to_string()}")
print(f"Extent (km) : W={extent[0]/1e3:.1f}  E={extent[1]/1e3:.1f}  "
      f"S={extent[2]/1e3:.1f}  N={extent[3]/1e3:.1f}")

## Part II: Visualize the pre- and post-event images

Before any processing, look at the data. The Mojave Desert has excellent optical coherence: sparse vegetation, stable sandy and rocky surfaces, and low cloud cover. These properties make it ideal for pixel tracking.

The surface rupture of the M7.1 mainshock is visible in the post-event image as a linear brightness change caused by freshly disturbed desert pavement.


In [ ]:
def percentile_stretch(arr, lo=2, hi=98):
    """Clip and normalise an array to [0, 1] for display."""
    vmin, vmax = np.nanpercentile(arr, [lo, hi])
    return np.clip((arr - vmin) / (vmax - vmin), 0, 1)


fig, axes = plt.subplots(1, 2, figsize=(13, 6), sharex=True, sharey=True)

for ax, img, title in zip(
    axes,
    [pre, post],
    ["Pre-event  (2019-06-24)", "Post-event (2019-07-14)"],
):
    ax.imshow(
        percentile_stretch(img),
        cmap="gray",
        extent=[e / 1e3 for e in extent],
        origin="upper",
        interpolation="none",
    )
    ax.set_title(title)
    ax.set_xlabel("Easting (km)")

axes[0].set_ylabel("Northing (km)")
fig.suptitle(
    "Sentinel-2 Band 8 (NIR, 10 m) — Ridgecrest, CA",
    fontsize=13,
)
plt.tight_layout()
plt.show()

print("\nCan you see the M7.1 surface rupture in the post-event image?")
print("It runs roughly NW–SE through the centre of the scene.")

> **Question 2.1:** Describe any visible differences between the pre- and post-event images. Where do you see evidence of surface change? What direction does the rupture trend?

> **Question 2.2:** Why is Band 8 (near-infrared) preferred for pixel tracking over desert terrain compared to a visible band?

### Why NIR for desert pixel tracking?

The near-infrared band (Band 8, 0.84–0.88 µm) is preferred for pixel tracking in desert environments for two reasons:

1. **Vegetation contrast.** Sparse desert shrubs reflect NIR strongly relative to bare rock and sand. This creates stable high-contrast texture that the cross-correlation can lock onto.
2. **Atmospheric scattering.** Longer wavelengths scatter less in the atmosphere than visible blue or green light, giving a cleaner surface signal.

For a co-seismic application like this one, both images are cloud-free Mojave summer scenes, so atmospheric differences are minimal. The main advantage here is the texture.


## Part III: Coregistration check

Pixel tracking measures the **difference** in position of surface features between two images. Any apparent displacement that exists everywhere in the scene — not just near the fault — is likely a residual misregistration or orbital error, not real ground motion.

We check coregistration by running the same cross-correlation on a **stable reference region** far from the fault. If the two images are well aligned, the recovered offset should be near zero everywhere in the reference region.


In [ ]:
# --- Define a stable reference region far from the rupture ---
# Adjust row/col indices if needed once you have seen the images.
# This corner region should be well away from the fault trace.
REF_ROWS = slice(50, 250)
REF_COLS = slice(50, 250)

pre_ref  = pre[REF_ROWS, REF_COLS]
post_ref = post[REF_ROWS, REF_COLS]

shift_ref, error_ref, _ = phase_cross_correlation(
    pre_ref, post_ref, upsample_factor=100
)

az_bias_m = -shift_ref[0] * pixel_size_m   # row shift → northward metres
rg_bias_m = -shift_ref[1] * pixel_size_m   # col shift → eastward metres

print("Coregistration check (reference region)")
print("-" * 40)
print(f"Azimuth (N–S) offset : {az_bias_m:+.3f} m")
print(f"Range   (E–W) offset : {rg_bias_m:+.3f} m")
print()
if abs(az_bias_m) < 1.0 and abs(rg_bias_m) < 1.0:
    print("✓  Residual offset < 1 m in both directions. Images are well aligned.")
else:
    print("⚠  Residual offset > 1 m — consider applying a bulk correction.")
    print(f"   Subtracting az_bias={az_bias_m:.3f} m, rg_bias={rg_bias_m:.3f} m")
    print("   from all profile measurements below.")

> **Question 3.1:** If the reference-region offset were 2 m rather than near-zero, what would that imply about the pre- and post-event images? How would you correct for it?

> **Question 3.2:** Would you expect better or worse coregistration between two images from the same Sentinel-2 pass direction (e.g. both descending) compared to one ascending and one descending? Why?


## Part IV: Sub-pixel cross-correlation

### How pixel tracking works

We divide the image into small overlapping **chips** (windows). For each chip we:

1. Extract the same-location chip from the pre- and post-event images.
2. Compute the cross-correlation between the two chips.
3. Find the peak of the correlation — its offset from centre gives the displacement of that chip.

We use `phase_cross_correlation` from `scikit-image`, which works in the Fourier domain and can resolve displacements to a small fraction of a pixel using the `upsample_factor` parameter.

**Sign convention:**
- Positive azimuth offset → the surface moved **northward** between the two images.
- Positive range offset → the surface moved **eastward**.

For a right-lateral rupture on a NW-trending fault:
- The NE block (right side looking NW along the fault) should move **SE** (negative azimuth, positive range).
- The SW block should move **NW** (positive azimuth, negative range).


In [ ]:
def correlate_chip(pre_img, post_img, row, col, chip=64, upsample=20):
    """
    Estimate sub-pixel displacement for one image chip.

    Parameters
    ----------
    pre_img, post_img : numpy.ndarray
        Full pre- and post-event images.
    row, col : int
        Centre pixel of the chip.
    chip : int
        Half-width of the chip in pixels.
    upsample : int
        Upsampling factor for sub-pixel precision.

    Returns
    -------
    az_m, rg_m : float
        Azimuth (N+) and range (E+) displacement in metres.
        Returns (nan, nan) if the chip contains NaN or has no variance.
    """
    half = chip // 2
    r0, r1 = row - half, row + half
    c0, c1 = col - half, col + half

    if r0 < 0 or r1 > pre_img.shape[0] or c0 < 0 or c1 > pre_img.shape[1]:
        return np.nan, np.nan

    pre_chip  = pre_img[r0:r1, c0:c1]
    post_chip = post_img[r0:r1, c0:c1]

    if np.any(np.isnan(pre_chip)) or np.any(np.isnan(post_chip)):
        return np.nan, np.nan
    if pre_chip.std() < 1e-6 or post_chip.std() < 1e-6:
        return np.nan, np.nan

    shift, _, _ = phase_cross_correlation(
        pre_chip, post_chip, upsample_factor=upsample
    )
    az_m = -shift[0] * pixel_size_m   # row → N (sign flip: +row = south)
    rg_m = -shift[1] * pixel_size_m   # col → E
    return az_m, rg_m

## Part V: Displacement profile across the fault

We extract a single fault-perpendicular profile — a row of chips sampled at regular intervals across the rupture zone. The fault trends NW–SE at roughly 315°, so a W–E profile is approximately perpendicular to it.

The profile row is set near the centre of the scene where the M7.1 rupture is best expressed. Adjust `PROFILE_ROW` if needed after inspecting the images in Part II.


In [ ]:
# --- Profile parameters ---
CHIP_SIZE  = 64    # chip side length in pixels (640 m)
STRIDE     = 20    # spacing between chip centres in pixels (200 m)
UPSAMPLE   = 20    # sub-pixel precision factor (0.5 m at 10 m pixels)

# Row through the middle of the scene — adjust if the rupture is off-centre
PROFILE_ROW = pre.shape[0] // 2

half = CHIP_SIZE // 2
col_centres = range(half, pre.shape[1] - half, STRIDE)

az_offsets_m = []
rg_offsets_m = []

for col in col_centres:
    az, rg = correlate_chip(
        pre, post, PROFILE_ROW, col,
        chip=CHIP_SIZE, upsample=UPSAMPLE
    )
    az_offsets_m.append(az)
    rg_offsets_m.append(rg)

col_centres  = np.array(list(col_centres))
az_offsets_m = np.array(az_offsets_m)
rg_offsets_m = np.array(rg_offsets_m)

# Convert pixel column index to easting (km) for plotting
easting_km = (extent[0] + col_centres * pixel_size_m) / 1e3

print(f"Profile row        : {PROFILE_ROW}  "
      f"(northing ≈ {(extent[3] - PROFILE_ROW * pixel_size_m)/1e3:.1f} km)")
print(f"Number of chips    : {len(col_centres)}")
print(f"Valid measurements : {np.sum(~np.isnan(az_offsets_m))}")
print(f"Azimuth range      : {np.nanmin(az_offsets_m):.1f} to "
      f"{np.nanmax(az_offsets_m):.1f} m")

In [ ]:
# --- Plot the displacement profile ---
fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)

# Subtract any residual bulk offset estimated from the reference region
az_corrected = az_offsets_m - az_bias_m
rg_corrected = rg_offsets_m - rg_bias_m

axes[0].plot(easting_km, az_corrected, "C0.", ms=4, label="Azimuth (N+)")
axes[0].axhline(0, color="k", lw=0.8, ls="--")
axes[0].set_ylabel("Displacement (m)")
axes[0].set_title("Azimuth offset  (northward = positive)")
axes[0].legend()

axes[1].plot(easting_km, rg_corrected, "C1.", ms=4, label="Range (E+)")
axes[1].axhline(0, color="k", lw=0.8, ls="--")
axes[1].set_ylabel("Displacement (m)")
axes[1].set_xlabel("Easting (km)")
axes[1].set_title("Range offset  (eastward = positive)")
axes[1].legend()

fig.suptitle(
    f"Fault-perpendicular displacement profile  (row {PROFILE_ROW})",
    fontsize=13,
)
plt.tight_layout()
plt.show()

> **Question 5.1:** Describe the shape of the azimuth offset profile. Is it consistent with right-lateral slip on a NW-trending fault? Explain which side moved which direction.

> **Question 5.2:** The range (E–W) profile should also show a step. Is the step in the same direction as you would expect for right-lateral motion? What does the combination of azimuth and range offsets tell you about the slip vector?

> **Question 5.3:** Are there any chips far from the fault with anomalously large offsets? What might cause those outliers?


## Part VI: Estimating total slip

The total surface slip is the **step** in the displacement profile across the fault — the difference between the mean offset on the NE block and the mean offset on the SW block.

We identify the fault location by finding where the azimuth gradient is steepest, then compute block means on either side.


In [ ]:
# --- Locate the fault: maximum gradient in the azimuth profile ---
valid = ~np.isnan(az_corrected)
az_valid  = az_corrected[valid]
east_valid = easting_km[valid]

# Smooth before differencing to reduce noise
smooth_az = uniform_filter(az_valid, size=5)
gradient  = np.abs(np.gradient(smooth_az))
fault_idx = np.argmax(gradient)
fault_km  = east_valid[fault_idx]

print(f"Fault located at easting ≈ {fault_km:.1f} km")

# --- Block means 10 km either side of the fault ---
BLOCK_KM = 10.0
sw_mask = (east_valid > fault_km - 2*BLOCK_KM) & (east_valid < fault_km - 2.0)
ne_mask = (east_valid > fault_km + 2.0) & (east_valid < fault_km + 2*BLOCK_KM)

az_sw = np.nanmean(az_valid[sw_mask])
az_ne = np.nanmean(az_valid[ne_mask])
az_step = az_ne - az_sw

rg_valid = rg_corrected[valid]
rg_sw = np.nanmean(rg_valid[sw_mask])
rg_ne = np.nanmean(rg_valid[ne_mask])
rg_step = rg_ne - rg_sw

total_slip = np.sqrt(az_step**2 + rg_step**2)

print(f"\nSlip components")
print("-" * 35)
print(f"Azimuth step (NE − SW) : {az_step:+.2f} m")
print(f"Range   step (NE − SW) : {rg_step:+.2f} m")
print(f"Total horizontal slip  : {total_slip:.2f} m")

# --- Annotate the profile plot ---
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(east_valid, az_valid, "C0.", ms=4, alpha=0.6)
ax.axvline(fault_km, color="r", lw=1.5, ls="--", label=f"Fault @ {fault_km:.1f} km")
ax.axhline(az_sw, xmin=0, xmax=(fault_km - east_valid[0]) / (east_valid[-1] - east_valid[0]),
           color="C2", lw=2, label=f"SW block mean = {az_sw:.2f} m")
ax.axhline(az_ne,
           xmin=(fault_km - east_valid[0]) / (east_valid[-1] - east_valid[0]),
           color="C3", lw=2, label=f"NE block mean = {az_ne:.2f} m")
ax.annotate(
    f"Step = {az_step:+.2f} m",
    xy=(fault_km, (az_sw + az_ne) / 2),
    xytext=(fault_km + 3, (az_sw + az_ne) / 2),
    arrowprops=dict(arrowstyle="->"),
    fontsize=10,
)
ax.set_xlabel("Easting (km)")
ax.set_ylabel("Azimuth offset (m)")
ax.set_title("Azimuth displacement profile with fault location and block means")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

> **Question 6.1:** The USGS and Caltech field teams measured peak surface slip of ~4–5 m on the M7.1 rupture. How does your estimate compare? List at least two reasons the optical pixel-tracking estimate might differ from field measurements.

> **Question 6.2:** The azimuth step and range step together define a 2D slip vector. Compute the azimuth of that vector (degrees clockwise from north) and comment on whether it is consistent with right-lateral slip on a NW-trending fault.

$$
\text{slip azimuth} = \arctan\!\left(\frac{\Delta_{\text{range}}}{\Delta_{\text{azimuth}}}\right)
$$


## Part VII: Why InSAR misses this signal

Sentinel-1 acquires SAR imagery from a descending orbit with an incidence angle of ~38° and a look azimuth of ~349° (nearly north, slightly west). The Ridgecrest M7.1 fault trends at approximately **315°** (NW–SE).

InSAR measures displacement in the **line-of-sight (LOS)** direction — a unit vector pointing from the ground to the satellite:

$$
\hat{\mathbf{l}} = \begin{pmatrix}
\sin\theta \sin\phi \\
\sin\theta \cos\phi \\
\cos\theta
\end{pmatrix}
$$

where $\theta$ is the incidence angle and $\phi$ is the look azimuth (from north, clockwise).

The fault-parallel unit vector (along the fault strike, in the direction of slip) is:

$$
\hat{\mathbf{f}} = \begin{pmatrix}
\sin\alpha \\
\cos\alpha \\
0
\end{pmatrix}
$$

where $\alpha$ is the fault strike. The InSAR sensitivity to fault-parallel horizontal slip is $\hat{\mathbf{l}} \cdot \hat{\mathbf{f}}$.


In [ ]:
# --- Sentinel-1 descending geometry ---
theta_deg = 38.0    # incidence angle
phi_deg   = 349.0   # look azimuth (from north, clockwise; descending right-looking)
alpha_deg = 315.0   # fault strike (NW–SE right-lateral)

theta = np.deg2rad(theta_deg)
phi   = np.deg2rad(phi_deg)
alpha = np.deg2rad(alpha_deg)

# LOS unit vector (E, N, Up)
l_hat = np.array([
    np.sin(theta) * np.sin(phi),
    np.sin(theta) * np.cos(phi),
    np.cos(theta),
])

# Fault-parallel unit vector (E, N, Up) — horizontal slip vector
f_hat = np.array([np.sin(alpha), np.cos(alpha), 0.0])

sensitivity = np.dot(l_hat, f_hat)

# For comparison: sensitivity to vertical
v_hat = np.array([0.0, 0.0, 1.0])
sensitivity_vert = np.dot(l_hat, v_hat)

print("Sentinel-1 descending LOS geometry")
print("-" * 45)
print(f"Incidence angle        : {theta_deg}°")
print(f"Look azimuth           : {phi_deg}°  (from N, CW)")
print(f"Fault strike           : {alpha_deg}°")
print()
print(f"LOS sensitivity to fault-parallel slip : {sensitivity:.3f}")
print(f"LOS sensitivity to vertical            : {sensitivity_vert:.3f}")
print()
print(f"If total slip = {total_slip:.2f} m (your estimate):")
print(f"  Expected InSAR LOS signal  = {abs(sensitivity * total_slip):.2f} m")
print(f"  Optical azimuth signal     ≈ {abs(az_step):.2f} m  (measured)")

In [ ]:
# --- Sensitivity as a function of fault strike ---
strikes = np.linspace(0, 360, 361)
sens_vals = [
    np.dot(l_hat, np.array([np.sin(np.deg2rad(s)), np.cos(np.deg2rad(s)), 0.0]))
    for s in strikes
]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(strikes, sens_vals, "C0", lw=1.8)
ax.axhline(0, color="k", lw=0.8, ls="--")
ax.axvline(alpha_deg, color="r", lw=1.5, ls="--",
           label=f"Ridgecrest fault (strike={alpha_deg}°) → sensitivity={sensitivity:.3f}")
ax.fill_between(strikes, sens_vals, 0,
                where=np.abs(sens_vals) < 0.1,
                alpha=0.2, color="orange",
                label="|sensitivity| < 0.1  (InSAR blind zone)")
ax.set_xlabel("Fault strike (° from N, CW)")
ax.set_ylabel("LOS sensitivity to fault-parallel slip")
ax.set_title("Sentinel-1 descending LOS sensitivity to horizontal strike-slip")
ax.legend(fontsize=9)
ax.set_xlim(0, 360)
plt.tight_layout()
plt.show()

> **Question 7.1:** Based on the sensitivity calculation, how large would the InSAR LOS signal be for the M7.1 rupture? Is this detectable? What fraction of the true slip would InSAR recover?

> **Question 7.2:** Look at the sensitivity curve. For what fault orientations is InSAR *most* sensitive to horizontal strike-slip motion? What kind of faults are those?

> **Question 7.3:** How could you combine ascending and descending InSAR tracks, together with optical pixel tracking, to fully resolve the 3D displacement vector?


## Part VIII: Extension — 2D displacement map

If you have time, compute the azimuth offset on a regular grid across the entire scene to produce a 2D map. This requires running the cross-correlation for every grid node — it will take a few minutes on Colab.

The output should show the classic "butterfly" pattern of a strike-slip earthquake: two lobes of opposite-sign displacement separated by the fault trace.


In [ ]:
# --- 2D displacement map (coarser grid to keep runtime reasonable) ---
GRID_STRIDE = 40     # pixels between grid nodes (400 m spacing)
CHIP_2D     = 64
UPSAMPLE_2D = 10

half2 = CHIP_2D // 2
row_nodes = range(half2, pre.shape[0] - half2, GRID_STRIDE)
col_nodes = range(half2, pre.shape[1] - half2, GRID_STRIDE)

n_rows = len(list(row_nodes))
n_cols = len(list(col_nodes))

az_map = np.full((n_rows, n_cols), np.nan)
rg_map = np.full((n_rows, n_cols), np.nan)

print(f"Computing {n_rows} × {n_cols} = {n_rows*n_cols} chips ...")

for i, row in enumerate(row_nodes):
    for j, col in enumerate(col_nodes):
        az, rg = correlate_chip(
            pre, post, row, col,
            chip=CHIP_2D, upsample=UPSAMPLE_2D
        )
        az_map[i, j] = az - az_bias_m
        rg_map[i, j] = rg - rg_bias_m
    if i % 5 == 0:
        print(f"  row {i+1}/{n_rows} done")

print("Done.")

In [ ]:
# --- Plot the 2D azimuth offset map ---
clim = np.nanpercentile(np.abs(az_map), 95)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, data, title in zip(
    axes,
    [az_map, rg_map],
    ["Azimuth offset  (northward = positive, m)",
     "Range offset  (eastward = positive, m)"],
):
    im = ax.imshow(
        data,
        cmap="RdBu_r",
        vmin=-clim, vmax=clim,
        extent=[e / 1e3 for e in extent],
        origin="upper",
        aspect="equal",
        interpolation="nearest",
    )
    plt.colorbar(im, ax=ax, label="m", shrink=0.7)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("Easting (km)")

axes[0].set_ylabel("Northing (km)")
fig.suptitle(
    "2D pixel-tracking displacement  —  Ridgecrest M7.1",
    fontsize=13,
)
plt.tight_layout()
plt.show()

> **Extension question:** Describe the spatial pattern of the azimuth offset map. Does it show the expected pattern for a right-lateral rupture? Where is the signal noisiest, and why might that be?


---
## Synthesis

> **In a short paragraph, summarise what you measured and what it tells you about the Ridgecrest M7.1 earthquake. Your response should:**
> 1. state your estimated total surface slip and compare it to published field measurements;
> 2. describe the 2D slip vector (azimuth and magnitude) and whether it is consistent with right-lateral faulting;
> 3. explain quantitatively why Sentinel-1 descending InSAR has low sensitivity to this rupture;
> 4. describe one situation where InSAR would be more useful than optical pixel tracking, and one where optical would be preferred.

**References**
- Barnhart, W. D., et al. (2019). Geodetic constraints on the 2019 Ridgecrest earthquake sequence. *Seismological Research Letters*, 91(4), 2056–2066. https://doi.org/10.1785/0220190322
- Leprince, S., et al. (2007). Automatic and precise orthorectification, coregistration, and subpixel correlation of satellite images. *IEEE TGRS*, 45(6), 1529–1558.
- Liu, C., et al. (2019). Three‐dimensional surface displacements of the 2019 Ridgecrest, California, earthquake sequence. *Geophysical Research Letters*, 46(22), 12764–12772.
- Sentinel-2 imagery: Copernicus Data Space Ecosystem, European Space Agency.


---

## Summary

- Optical pixel tracking measures surface displacement by cross-correlating image chips from pre- and post-event satellite images at sub-pixel precision.
- A coregistration check on a stable off-fault region is essential before interpreting any displacement signal.
- For the Ridgecrest M7.1, the azimuth offset profile shows a clear step across the NW-trending rupture consistent with right-lateral slip.
- The total horizontal slip estimated from azimuth and range offsets is comparable to published field measurements.
- Sentinel-1 descending InSAR has low sensitivity (~0) to fault-parallel horizontal motion on NW-trending faults because the LOS is nearly perpendicular to the slip direction.
- Optical pixel tracking in the image azimuth direction is sensitive to exactly the component InSAR misses, making the two techniques complementary.
- The 2D displacement map reveals the spatial pattern of co-seismic deformation across the entire scene.

This lab used Sentinel-2 Band 8 imagery processed for EPS 166 from the Copernicus Data Space Ecosystem. The pixel-tracking approach follows Leprince et al. (2007).
